### Task 7: 

Create an interactive model of the virtual, enlarged image of an object placed inside the focal range of an ideal thin lens.

Have not figured out how to run it in GitHub, and so - for the purposes of the submission video - this task was run in Thonny.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
import numpy as np
import os
from PIL import Image

def load_img(image_path): #loads image file and converts it to a numpy array
    try:
        img = Image.open(image_path) 
        return np.array(img)
    except Exception as e:
        print(f"Error loading image: {e}")
        return None
    
def open_img(): # opens image file and returns it as an array
    image_path = "stinkbug.png"   
    '''replace above image file if needed!'''
    if not os.path.exists(image_path): # error handling
        print(f"'{image_path}' not found")
        raise SystemExit
    else:
        img_path = load_img(image_path)
        if img_path is None: # more error handling
            print('uhoh')
            raise SystemExit
        print(f"{image_path} successfully loaded")
        return img_path

orig_img = open_img()

def calc_lens(X, Y, f): # calc transformation w/ lens equation
    xx = -(-1/X + 1/f)**(-1)
    yy = Y * xx / X
    return xx, yy

''''''
# scaling image up/down to fit graph
height, width = orig_img.shape[:2]
aspect_ratio = height / width
x0, y0 = 10 , 0 # object centre
obj_w = 5       # width of object (scales the image)
f = 15          # focal length
# maximum and minimum x and y coords
x_min = x0 - obj_w / 2
x_max = x0 + obj_w / 2
y_max = y0 + obj_w / 2 * aspect_ratio
y_min = y0 - obj_w / 2 * aspect_ratio
# meshgrid
x = np.linspace(0, obj_w, width) - obj_w/2 + x0
y = -np.linspace(0, obj_w*aspect_ratio, height) + obj_w*aspect_ratio/2 + y0
X, Y = np.meshgrid(x,y)
# figure, lens plot and focal point plot
fig, ax = plt.subplots(1, 1, figsize=(12, 8))
thin_lens = Ellipse(xy=(0, 0),  label = 'Thin Lens', width=3, height=20, edgecolor='b', fc = 'None', lw=0.5)
ax.add_patch(thin_lens)
marker = plt.plot(0,0, marker='*', color='red',
         markersize=3, zorder = 5, label='Lens Focal Point')
# real image
img_show = ax.pcolormesh(X, Y, orig_img, shading='auto', zorder=3)
#transformed/virtual image
xx,yy = calc_lens(X, Y, f)
virtual_show = ax.pcolormesh(xx, yy, orig_img, shading='auto', zorder=3)
# axis position, limits and labels
ax.spines['left'].set_position(('outward', 0.8)) # position of y-axis
ax.spines['bottom'].set_position(('outward', 0.8)) # position of x-axis
ax.set_xlim(-5, 100)
ax.set_ylim(-45, 45)
ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('y', fontsize=12)
# title and text 
ax.set_title('Thin Converging Lens: Inside Focal Range', fontsize=16)
plt.suptitle('focal length = 15', color = 'grey', fontsize=10)
obj_txt = ax.text(10, -6, 'Real Object', 
            ha='center', fontsize=10)
virtual_txt = ax.text(42.5, -12.5, 'Virtual Object', 
            ha='center', fontsize=10)
plt.text(100.5,43.5, 'Use Your Arrow Keys to Move the Image!', fontsize = 8)
uhoh_txt = ax.text(100.5,40, '', color = 'red', fontsize = 8)
# grid
ax.grid(True, alpha=0.5, linestyle='-', zorder = 1)
ax.set_aspect('equal', adjustable='box')
# legend
plt.legend(loc='upper left', fontsize=12)
''''''

def update(event): # update image position and warning text
    global X, Y, x_min, x_max, y_max, y_min
    uhoh_txt.set_text('')
    obj_txt.set_text('')
    virtual_txt.set_text('')
    fig.canvas.draw_idle()
    # moving up
    if event.key == 'up':
        y_max += 0.5
        if y_max > 10: # top of lens y-coord == 10
            y_max -= 0.5
            uhoh_txt.set_text('Out of Bounds')
        else:
            Y += 0.5
            y_min += 0.5
            move_pic()
    #moving down
    elif event.key == 'down':
        y_min -= 0.5
        if y_min < -10: # bottom of lens y-coord == -10
            y_min += 0.5
            uhoh_txt.set_text('Out of Bounds')
        else:
            Y -= 0.5
            y_max -= 0.5
            move_pic()
    #moving left
    elif event.key == 'left':
        x_min -= 0.5
        if x_min <= 1: # right of lens x-coord == 1
            x_min += 0.5
            uhoh_txt.set_text('Hit the lens!')
        else:
            X -= 0.5
            x_max -= 0.5
            move_pic()
    #moving right
    elif event.key == 'right':
        x_max += 0.5
        if x_max >= f: # f == focal length
            x_max -= 0.5
            uhoh_txt.set_text('Going out of focal range!')
        else:
            X += 0.5
            x_min += 0.5
            move_pic()
    fig.canvas.draw_idle()
 
def move_pic(): # actually move the images
    global img_show, virtual_show
    xx,yy = calc_lens(X, Y, f) # re-calculate virtual img shape
    img_show.remove()
    virtual_show.remove()
    img_show = ax.pcolormesh(X, Y, orig_img, shading='auto', zorder=3)
    virtual_show = ax.pcolormesh(xx, yy, orig_img, shading='auto', zorder=3)
    plt.show()

fig.canvas.mpl_connect('key_press_event', update)
plt.show()